# 第二步:教师蒸馏 + 正式训练 · Skincare Advisor

**可以一键跑**(`代码执行程序 → 全部运行`)。这个 notebook 设计成**两轮一键跑**:

| 轮次 | 设置 | 会发生什么 |
|---|---|---|
| **第 1 轮** | `APPROVE_FULL_RUN = False`(默认) | 跑到小样本诊断为止,**自动停下**。花费约 20 次调用 |
| **第 2 轮** | 看完结果满意 → 改成 `True` → 再次全部运行 | 全量蒸馏 + SFT + GRPO + 自动存 Drive |

**为什么要两轮**:C 段之后是花钱的分水岭。质量不达标就上量,
等于拿 800 条烂数据训模型 —— 钱白花,模型更差。第 1 轮的闸门会自动拦住。

**闸门是自动的**:通过率低于阈值会直接抛异常终止后续格子,你不需要盯着。

**中途会停下来等你两次**(都在最开头):上传代码 zip、授权 Google Drive。
授权完之后就可以走开了。

> 🔐 API key 用 Colab Secrets(左侧 🔑 图标),名称 `OPENAI_API_KEY`,
> 记得打开「笔记本访问权限」开关。


## 0. 配置 —— 只改这一格


In [ ]:
# ================= 一键跑的总开关 =================
APPROVE_FULL_RUN = False   # 第 1 轮保持 False;看完 C 段结果满意后改 True 再全部运行

MIN_PASS_RATE = 0.70       # 通过率低于此值,闸门会拦住不让上量
# ==================================================

MODEL   = 'gpt-4o-mini'                  # 教师模型
BASE    = 'Qwen/Qwen2.5-1.5B-Instruct'   # OOM 就换 Qwen/Qwen2.5-0.5B-Instruct
N_SMALL = 20                             # 小样本试跑条数
N_FULL  = 800                            # 全量蒸馏条数
USE_DRIVE = True                         # 把模型与数据存到 Google Drive

SFT_EPOCHS, GRPO_STEPS = 2, 300

print(f"APPROVE_FULL_RUN = {APPROVE_FULL_RUN}")
print('第 1 轮:跑到 C 段诊断后自动停' if not APPROVE_FULL_RUN else '第 2 轮:将执行全量蒸馏与训练')


## A. 环境 + 代码 + Drive(所有需要你点击的操作都在这一格)


In [ ]:
import os, sys, importlib, importlib.util, shutil, zipfile
from pathlib import Path

WORK = Path('/content/skincare')

# --- 代码就位(会话重启过就要重传)---
if not (WORK/'pyproject.toml').exists():
    from google.colab import files
    print('请上传 skincare_for_colab.zip …')
    up = files.upload(); name = list(up)[0]
    tmp = Path('/content/_unzip')
    if tmp.exists(): shutil.rmtree(tmp)
    with zipfile.ZipFile(name) as z: z.extractall(tmp)
    root = next(p.parent for p in tmp.rglob('pyproject.toml'))
    if WORK.exists(): shutil.rmtree(WORK)
    shutil.move(str(root), str(WORK))
else:
    print('代码已在 /content/skincare')

os.chdir(WORK)
sys.path[:0] = [str(WORK), str(WORK/'src')]
os.environ['PYTHONPATH'] = f"{WORK}:{WORK/'src'}"

# --- 依赖 ---
os.system('pip install -q openai transformers peft trl datasets accelerate '
          'fastapi pydantic python-multipart python-dotenv pandas pytest httpx pyyaml 2>&1 | tail -2')
if importlib.util.find_spec('torchao') is not None:
    print('卸载旧版 torchao(与新版 PEFT 冲突)')
    os.system('pip uninstall -y -q torchao'); importlib.invalidate_caches()

# --- Drive(现在授权,后面训练完就能自动保存)---
DRIVE_OK = False
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        MODEL_DST = Path('/content/drive/MyDrive/skincare_models'); MODEL_DST.mkdir(parents=True, exist_ok=True)
        DATA_DST  = Path('/content/drive/MyDrive/skincare_data');   DATA_DST.mkdir(parents=True, exist_ok=True)
        DRIVE_OK = True
        print(f'Drive 已挂载 -> {MODEL_DST}')
    except Exception as e:
        print(f'⚠️ Drive 挂载失败({e}),训练结果只能手动下载')

# --- 防呆:确认这份代码带着 notebook 需要的功能(避免用了旧的 zip)---
_db = (WORK/'src'/'skincare'/'llm'/'data_build.py').read_text()
_gp = (WORK/'src'/'skincare'/'llm'/'grpo_train.py').read_text()
_missing = []
if '"--inspect"' not in _db:        _missing.append('data_build.py 的 --inspect 参数')
if 'distill_summary' not in _db:    _missing.append('data_build.py 的 distill_summary 输出(闸门要用)')
if '_precision' not in _gp:         _missing.append('grpo_train.py 的显卡精度自适应')
if 'is_trainable=True' not in _gp:  _missing.append('grpo_train.py 的 SFT adapter 加载')
if _missing:
    raise SystemExit(
        '❌ 这份代码是旧版本,缺少:\n  - ' + '\n  - '.join(_missing) +
        '\n\n请重新打包上传:在 Mac 上删掉旧的 skincare_for_colab.zip,'
        '\n让 Claude 重新生成,或自己执行:'
        '\n  cd ~/Documents && rm -f skincare/skincare_for_colab.zip'
        '\n  zip -r skincare_for_colab.zip skincare -x "*.git/*" "*__pycache__*" "*.venv*"'
        '\n然后删掉 /content/skincare 重跑本格。')
print('✅ 代码版本检查通过')

import torch
gpu = torch.cuda.is_available()
print(f"\nGPU {gpu} | 精度 {'bf16' if gpu and torch.cuda.is_bf16_supported() else ('fp16' if gpu else 'fp32')}")
assert (WORK/'pyproject.toml').exists(), '代码没就位'
assert gpu, '没有 GPU:代码执行程序 -> 更改运行时类型 -> T4 GPU'
print('✅ A 段完成')



## B. 验证 API key(只花 1 次调用)


In [ ]:
try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
except Exception as e:
    raise SystemExit(f'读不到 Secrets 里的 OPENAI_API_KEY: {e}\n'
                     '左侧 🔑 -> 添加新密钥 -> 名称 OPENAI_API_KEY -> 打开笔记本访问权限')

from openai import OpenAI
from skincare.llm.data_build import build_rows
from skincare.llm.prompts import SYSTEM
from skincare.llm.rewards import reward_breakdown

row = build_rows(1, mock=True)[0]
r = OpenAI().chat.completions.create(
    model=MODEL, temperature=0.2, response_format={'type': 'json_object'},
    messages=[{'role':'system','content':SYSTEM}, {'role':'user','content':row['prompt']}])
out = r.choices[0].message.content

print('=== 教师返回(节选)==='); print(out[:700])
b = reward_breakdown(out, concerns=row['concerns'], evidence_ids=row['evidence_ids'],
                     product_ids=row['product_ids'], pregnant=row['pregnant'], avoid=row['avoid'])
print('\n=== 奖励分解 ===')
for k, v in b.items(): print(f'  {k:18s} {v:.2f}')
print(f"\n用量:输入 {r.usage.prompt_tokens} tok,输出 {r.usage.completion_tokens} tok")
assert b['format'] > 0.5, '教师没返回合法 JSON —— 先修 prompts.py,别往下走'
print('\n✅ B 段完成 —— key 可用,返回格式正确')


## C. 小样本蒸馏 + 质量诊断

`--inspect 3` 会打印保留样本和被丢弃样本的原文,**请肉眼读一读**:
推荐理由说得通吗?引用了证据吗?成分对得上关注点吗?


In [ ]:
!python -m skincare.llm.data_build --n $N_SMALL --mock-retrieval --mode sft --inspect 3 2>&1 | tail -60


## 🚦 闸门 —— 自动判断能否上量


In [ ]:
import json
from pathlib import Path

S = json.loads(Path('data/processed/distill_summary.json').read_text())
rate = S['pass_rate']

print(f"通过率 {rate:.0%}  (保留 {S['kept']}/{S['total']},阈值 {S['threshold']})")
print('\n各分量均值:')
for k, v in S['component_means'].items():
    print(f"  {k:18s} {v:.2f}  {'█'*int(v*20)}")
if S.get('usage', {}).get('calls'):
    u = S['usage']; n = u['calls']
    print(f"\n本次 {n} 次调用:输入 {u['prompt_tokens']:,} / 输出 {u['completion_tokens']:,} tok")
    print(f"外推 {N_FULL} 条:输入约 {u['prompt_tokens']//n*N_FULL:,} / "
          f"输出约 {u['completion_tokens']//n*N_FULL:,} tok(按当前单价自行折算)")

print('\n' + '='*56)
if rate < MIN_PASS_RATE:
    raise RuntimeError(
        f"闸门拦截:通过率 {rate:.0%} < 门槛 {MIN_PASS_RATE:.0%}\n"
        f"最弱环节是 [{S['weakest']}]。别急着上量,先看上一格的诊断建议:\n"
        f"  · ingredient_match 低 -> 补 data/knowledge/ingredient_rules.json(每个关注点 8-12 个成分)\n"
        f"  · grounding 低       -> prompts.py 里把'必须引用 evidence_id'写得更硬\n"
        f"  · format 低          -> prompts.py 的 JSON schema 说明不够明确\n"
        f"改完重跑 C 段。")

if not APPROVE_FULL_RUN:
    raise SystemExit(
        f"✅ 质量达标(通过率 {rate:.0%})。\n\n"
        f"这是第 1 轮,到此为止 —— 请翻上去读 C 段打印的教师答案,确认质量。\n"
        f"满意的话:把第 0 格的 APPROVE_FULL_RUN 改成 True,再点一次「全部运行」。\n"
        f"(小样本的结果有缓存,第 2 轮不会重复付费)")

print(f'✅ 闸门放行(通过率 {rate:.0%},已授权全量运行)')


## D. 全量蒸馏 → SFT → GRPO(每步完成立即存 Drive)


In [ ]:
# D1. 全量蒸馏(带缓存,断了重跑不会重复烧钱)
!python -m skincare.llm.data_build --n $N_FULL --mock-retrieval --mode both 2>&1 | tail -28

import shutil
from pathlib import Path
if DRIVE_OK:
    for f in Path('data/processed').glob('*.jsonl'):
        shutil.copy(f, DATA_DST/f.name)
    for f in Path('data/processed').glob('*.json'):
        shutil.copy(f, DATA_DST/f.name)
    print(f'\n✅ 数据已备份到 {DATA_DST}(蒸馏花过钱,别弄丢)')


In [ ]:
# D2. 正式 SFT
os.environ['BASE'] = BASE
SFT_OUT = '/content/skincare/models/llm/sft-lora'
!python -m skincare.llm.sft_lora --base $BASE --epochs $SFT_EPOCHS --bs 1 --accum 8 \
    --max-len 2048 --out $SFT_OUT 2>&1 | tail -12

from pathlib import Path
import shutil
assert any(Path(SFT_OUT).glob('adapter*')), 'SFT 没产出 adapter'
if DRIVE_OK:
    shutil.copytree(SFT_OUT, MODEL_DST/'sft-lora', dirs_exist_ok=True)
    print(f'✅ SFT adapter 已存到 {MODEL_DST}/sft-lora')


In [ ]:
# D3. 正式 GRPO(从 SFT adapter 继续)
# 重点看 rewards/xxx_reward/mean 五行是否随步数上升 —— 那就是报告要的曲线
GRPO_OUT = '/content/skincare/models/llm/grpo'
!python -m skincare.llm.grpo_train --base $BASE --adapter $SFT_OUT \
    --steps $GRPO_STEPS --group-size 8 --accum 4 --max-completion-length 512 \
    --out $GRPO_OUT 2>&1 | tail -22

assert any(Path(GRPO_OUT).glob('adapter*')), 'GRPO 没产出 adapter'
if DRIVE_OK:
    shutil.copytree(GRPO_OUT, MODEL_DST/'grpo', dirs_exist_ok=True)
    print(f'✅ GRPO adapter 已存到 {MODEL_DST}/grpo')


## E. 更新 manifest —— 交给组员 C 做评估的唯一接口


In [ ]:
import json
from pathlib import Path

mf = Path('/content/skincare/models/llm/manifest.json')
m = json.loads(mf.read_text())
m['base'] = BASE
m['sft']  = str(MODEL_DST/'sft-lora') if DRIVE_OK else SFT_OUT
m['grpo'] = str(MODEL_DST/'grpo')     if DRIVE_OK else GRPO_OUT
mf.write_text(json.dumps(m, ensure_ascii=False, indent=1))
if DRIVE_OK: shutil.copy(mf, MODEL_DST/'manifest.json')
print(mf.read_text())

print('\n' + '='*56)
print('  全部完成')
print('='*56)
print(f'\n交给组员 C:{MODEL_DST if DRIVE_OK else "models/llm/"} 下的')
print('  sft-lora/ · grpo/ · manifest.json')
print('\n她跑这条命令就能出报告要的三段式对比表:')
print('  python -m skincare.eval.run_eval --split data/processed/rl_test.jsonl \\')
print('         --variants base sft grpo')
